
# Thesis-ready multi-seed harmful-edge mining and PRBCD evaluation

This notebook runs the **entire stochastic pipeline independently for every seed**:

1. dataset split and victim-model training,
2. PRBCD/random candidate generation,
3. harmfulness-label mining,
4. LP-GNN training,
5. model-guided PRBCD attack,
6. aggregation across seeds.

The LP-GNN trained for seed \(s\) is used only in the attack for seed \(s\). Candidate labels, models, or attack outputs are never mixed across seeds. Aggregation is performed only after complete independent runs.

Place `multiseed_experiment.py` in the project root or next to this notebook before running.


In [ ]:

import os
import sys
from pathlib import Path

import config

# Run from the repository root, matching the original notebook.
%cd {config.PROJECT_DIR}

%load_ext autoreload
%autoreload 2
%matplotlib inline

from IPython.display import display

from multiseed_experiment import (
    AttackSweepSpec,
    PipelineConfig,
    aggregate_attack_results,
    generate_all_seed_artifacts,
    plot_mean_accuracy_with_ci,
    run_paired_attack_sweep,
)

print("Project directory:", Path.cwd())



## 1. Experiment configuration

Five seeds are a reasonable minimum for a thesis experiment; ten seeds are preferable when runtime allows. Keep the seed list fixed before producing the final tables.


In [ ]:

SEEDS = (0, 1, 2, 3, 4)

PIPELINE_CONFIG = PipelineConfig(
    # Dataset and split
    dataset="cora_ml",
    seeds=SEEDS,
    train_ratio=0.15,
    val_ratio=0.10,
    test_ratio=0.30,

    # Victim model
    model_name="GCN",
    model_label="GCN",
    victim_dropout=0.5,
    victim_lr=1e-2,
    victim_weight_decay=1e-3,
    victim_patience=300,
    victim_max_epochs=3000,

    # Use a dedicated storage type so every attack loads the matching
    # seed-specific victim model generated by this notebook.
    model_storage_type="thesis_multiseed",

    # Candidate generation
    use_prbcd_candidates=True,
    prbcd_candidate_fraction=0.5,
    n_prbcd_runs=6,
    prbcd_budget_fraction=0.5,
    prbcd_epochs=5,
    prbcd_search_space=200_000,
    prbcd_n_epochs_resampling=1,

    # Harmfulness mining
    mining_mode="endpoint",
    n_subsets=1000,
    subset_fraction=0.05,
    balance_ratio=1.0,

    # LP-GNN
    label_mode="continuous",
    extreme_fraction=0.2,
    lp_num_epochs=500,
    lp_hidden_dim=64,
    lp_out_dim=64,
    lp_lr=5e-4,
    lp_weight_decay=5e-4,
    lp_train_ratio=0.70,
    lp_val_ratio=0.15,
    lp_test_ratio=0.15,
    lp_min_epochs_before_early_stop=300,
    lp_early_stop_patience=15,
    lp_early_stop_min_delta=1e-4,

    # Storage and reproducibility
    data_dir="./data",
    artifact_dir="cache",
    output_root="extendedPlotting/multiseed",
    deterministic=True,
    device="cpu",
    data_device="cpu",
)

PIPELINE_CONFIG.validate()
PIPELINE_CONFIG



## 2. Generate all seed-specific artifacts

Each seed gets separate cache paths and a configuration/candidate fingerprint. Re-running this cell reuses compatible mining caches, but trains a fresh in-memory LP-GNN for each seed. Set `force_mining=True` only when the mining implementation changed and old caches must be invalidated.


In [ ]:

SEED_ARTIFACTS = generate_all_seed_artifacts(
    PIPELINE_CONFIG,
    force_mining=False,
    verbose=True,
)

GENERATION_DIR = Path(PIPELINE_CONFIG.output_root) / "generation"

generation_by_seed = __import__("pandas").read_csv(
    GENERATION_DIR / "generation_metrics_by_seed.csv"
)
generation_aggregate = __import__("pandas").read_csv(
    GENERATION_DIR / "generation_metrics_aggregate.csv"
)

display(generation_by_seed)
display(generation_aggregate)



Do **not** average the mined edge-label vectors or LP-GNN parameters across seeds: candidate pools and victim decision boundaries differ. Report the distribution of generation metrics and average only comparable evaluation quantities.


## 3. Define the seed-paired attack sweep

In [ ]:

ATTACK_SPEC = AttackSweepSpec(
    experiment_grid={
        "attack": ["PRBCD"],
        "semi": [True],
        "epsilon": [0.05],
        "use_cert": [
            "accuracy_drop_selector",
            "none",
            "accuracy_drop_selector_with_resampling",
        ],
        # Deliberately omitted: seed is injected from SEED_ARTIFACTS.
    },
    attack_grid={
        "block_size": [4000],
        "epochs": [100],
        "fine_tune_epochs": [50],
        "keep_heuristic": ["WeightOnly"],
        "do_synchronize": [True],
        "loss_type": ["tanhMargin"],
    },
    selector_grid={
        "accuracy_drop_selector_mode": ["one_sample"],
        "n_candidates_k_sample": [2_000],
        "n_candidates_one_sample": [5_000],
        "drop_mode": ["endpoint"],
        "acc_drop_threshold_k_samples": [1e-3],
        "loss_drop_threshold_k_samples": [1e-3],
        "k_samples_batch": [10],
        "training_data_node_cap": [15],
        "tau": [0.8],
        "score_batch_size": [1_000],
        "max_sampling_tries": [2_000_000],
        "exclude_tried": [True],
    },
    selector_fixed={
        "lp_hit_rate_detour": False,
        "lp_hit_rate_top_k": 200,
        "lp_hit_rate_out_dir": "extendedPlotting/lpEndpointHitRate",
    },
)



## 4. Run attacks

The runner writes one JSONL record immediately after every completed run, so partial progress survives a crash. `resume=True` skips already successful effective configurations in the selected output directory.


In [ ]:

ATTACK_OUTPUT_DIR = run_paired_attack_sweep(
    PIPELINE_CONFIG,
    SEED_ARTIFACTS,
    ATTACK_SPEC,
    stop_on_error=False,
    save_raw_results=False,
    resume=True,
)

print("Attack output directory:", ATTACK_OUTPUT_DIR)


## 5. Aggregate across seeds

In [ ]:

AGGREGATED = aggregate_attack_results(ATTACK_OUTPUT_DIR)

print("Final metrics by seed")
display(AGGREGATED["summary"])

print("Final mean, sample SD, SEM, and 95% CI")
display(AGGREGATED["summary_aggregate"])

print("Paired final differences against the random/no-selector baseline")
display(AGGREGATED["paired_final_aggregate"])

print("Per-epoch mean and 95% CI")
display(AGGREGATED["epochs_aggregate"].head(20))

PLOT_PATHS = plot_mean_accuracy_with_ci(
    ATTACK_OUTPUT_DIR,
    include_baseline_epoch=True,
)
print(f"Saved {len(PLOT_PATHS)} mean/CI plot(s).")
for path in PLOT_PATHS:
    print(" -", path)



## 6. Interpretation for the thesis

The main attack outcome is `final_accuracy_mean ± final_accuracy_std`, accompanied by the 95% confidence interval and the number of completed seeds.

The stronger comparison is the **paired seed difference**

\[
\Delta_s = \mathrm{accuracy}_{\text{none},s}
           - \mathrm{accuracy}_{\text{guided},s}.
\]

A positive value means the guided initializer produced lower victim accuracy than the `none` baseline for the same split, victim model, and outer seed. Report the mean and sample standard deviation of \(\Delta_s\), and retain all per-seed values in an appendix or supplementary table.

The generated folder also contains:

- `generation_metrics_by_seed.csv` and `generation_metrics_aggregate.csv`,
- `summary_by_seed.csv` and `summary_aggregate.csv`,
- `epochs_by_seed.csv` and `epochs_aggregate.csv`,
- paired baseline-difference tables,
- a configuration/environment manifest,
- mean curves with 95% confidence bands.
